# SmolVLM2 TinyDoc QLoRA (Kaggle)\n\nQLoRA fine-tune of `HuggingFaceTB/SmolVLM2-2.2B-Instruct` on REAL document pairs only (docvqa/sroie/docmatix/ocrbench/funsd, 46k). No synthetic templates.\n\nPivot from TinyDoc-VLM-768 (dead vision tower — see repo docs). Vision tower stays frozen (it works); LoRA adapters learn document tasks.\n\nAdapter pushed to `eulogik/SmolVLM2-TinyDoc-real`.

In [ ]:
import subprocess, sys, os, time

# Single GPU only: DataParallel (2xT4) is broken with PEFT (StopIteration
# in SmolVLMModel.dtype on replicas). T4 has no bf16 -> script uses fp16.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

REPO_URL = 'https://github.com/eulogik/TinyDoc-VLM'
REPO = '/kaggle/working/tinydoc-vlm'

DATA_REPO = os.environ.get('DATA_REPO', 'eulogik/TinyDoc-VLM-real-data')
HUB_ID = os.environ.get('HUB_ID', 'eulogik/SmolVLM2-TinyDoc-real')
MAX_STEPS = os.environ.get('MAX_STEPS', '3000')
BATCH = os.environ.get('BATCH', '2')
ACCUM = os.environ.get('ACCUM', '8')
LR = os.environ.get('LR', '0.0002')

if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        _c = UserSecretsClient()
        for _ in range(5):
            try:
                os.environ['HF_TOKEN'] = _c.get_secret('HF_TOKEN')
                break
            except Exception:
                time.sleep(5)
    except Exception:
        pass

if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN not set. Add it to Kaggle Secrets (Settings > Secrets).')

if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO])

# SmolVLM2 QLoRA stack (Kaggle base lacks trl/peft/bitsandbytes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.50', 'trl>=0.14', 'peft', 'bitsandbytes',
    'pillow', 'datasets', 'huggingface_hub', 'num2words'])

p = subprocess.run(
    [sys.executable, 'training/smolvlm2_qlora.py',
     '--data-repo', DATA_REPO, '--hub-id', HUB_ID,
     '--max-steps', MAX_STEPS, '--batch', BATCH,
     '--accum', ACCUM, '--lr', LR,
     '--model-path', '/kaggle/input/smolvlm2-2-2b-instruct'],
    cwd=REPO,
)
sys.exit(p.returncode)

